# eppur-fluctuat tutorial

This notebook outlines an **editable install**, preparing inputs, running the baseline **CPU AutoDock Vina** pipeline, inspecting outputs, and understanding the **QSAR ranking fallback** when linear models are untrusted.

**Prerequisites:** conda env from `environment.yml` (or pip + Open Babel on `PATH`). GPU flags in `config.yaml` do **not** enable GPU Vina.

## 1. Editable install

From the repository root:

In [ ]:
# pip install -e ".[dev]"
import sys
from pathlib import Path

ROOT = Path("..").resolve()  # if notebook cwd is examples/
if not (ROOT / "config.yaml").is_file():
    ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT / "scripts"))
import eppur_fluctuat
print("eppur-fluctuat version", eppur_fluctuat.__version__)

## 2. Inputs and config

Place `protein.pdb`, `ligands.csv` (`name`, `smiles`), and optional `curated_pIC50.csv` under `workspace/inputs/` (defaults in `config.yaml`). Adjust `protein.pocket_center` / `pocket_size` for your target.

## 3. Run the CLI pipeline

Run in terminal from `ROOT`:

```bash
python scripts/prepare_protein.py
python scripts/dock_ligands.py
python scripts/build_qsar.py
python scripts/generate_report.py
```

Inspect `workspace/outputs/docking_results.csv`, `predictions.csv`, and `qsar_report.html`.

## 4. Inspect docking helper API (optional)

`run_docking` wraps Vina after maps are computed — useful for tests or custom loops.

In [ ]:
from unittest.mock import MagicMock
from pathlib import Path
from dock_ligands import run_docking

v = MagicMock()
v.energies.return_value = [[[-8.0, 0, 0, 0, 0, 0]]]
out = Path("_tutorial_best.pdbqt")
run_docking(v, "dummy.pdbqt", 4, 3, out)
print("affinity", -8.0)

## 5. Fallback behaviour

When leave-one-out **R²** is below `qsar.fallback_loo_r2_threshold`, `build_qsar.py` sets `prediction_source` to `fallback_ranking` and orders compounds by docking score. To **force** exploration of this path in real data, temporarily raise the threshold in a scratch `config.yaml` or use a degenerate descriptor column (flat docking scores)—see `tests/test_qsar_fallback.py`.